In [1]:
pip install streamlit

Note: you may need to restart the kernel to use updated packages.


In [7]:
%%writefile app.py

UsageError: %%writefile is a cell magic, but the cell body is empty.


In [9]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

# ---------------- Page ----------------
st.set_page_config(
    page_title="Customer Churn Predictor",
    page_icon="📉",
    layout="centered"
)

# ---------------- Style (CSS) ----------------
st.markdown(
    """
    <style>
      .center {text-align: center;}
      .subtle {color: #6b7280; font-size: 0.95rem;}
      .card {
        border: 1px solid rgba(0,0,0,0.08);
        border-radius: 14px;
        padding: 18px 18px 10px 18px;
        background: rgba(255,255,255,0.02);
      }
      .result-card {
        border: 1px solid rgba(0,0,0,0.10);
        border-radius: 14px;
        padding: 16px;
        background: rgba(255,255,255,0.02);
      }
      div.stButton > button {
        border-radius: 12px;
        padding: 0.7rem 1.0rem;
        font-weight: 600;
      }
    </style>
    """,
    unsafe_allow_html=True
)

# ---------------- Header (Centered) ----------------
st.markdown('<h1 class="center">📉 Customer Churn Predictor</h1>', unsafe_allow_html=True)
st.markdown('<p class="center subtle">Enter customer details and click <b>Predict</b> to get churn probability.</p>', unsafe_allow_html=True)
st.write("")

# ---------------- Load Model ----------------
model = joblib.load("churn_model_pipeline.joblib")

# ---------------- Inputs (Organized) ----------------
st.markdown('<div class="card">', unsafe_allow_html=True)
st.subheader("Customer Information")

c1, c2 = st.columns(2)

with c1:
    gender = st.selectbox("Gender", ["Female", "Male"])
    senior = st.selectbox("Senior Citizen", [0, 1])
    partner = st.selectbox("Partner", ["Yes", "No"])
    dependents = st.selectbox("Dependents", ["Yes", "No"])

with c2:
    tenure = st.number_input("Tenure (months)", min_value=0, max_value=100, value=12)
    contract = st.selectbox("Contract", ["Month-to-month", "One year", "Two year"])
    paperless = st.selectbox("Paperless Billing", ["Yes", "No"])
    payment = st.selectbox(
        "Payment Method",
        ["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"]
    )

st.divider()
st.subheader("Services")

s1, s2 = st.columns(2)

with s1:
    phone_service = st.selectbox("Phone Service", ["Yes", "No"])
    multiple_lines = st.selectbox("Multiple Lines", ["No", "Yes", "No phone service"])
    internet_service = st.selectbox("Internet Service", ["DSL", "Fiber optic", "No"])
    online_security = st.selectbox("Online Security", ["No", "Yes", "No internet service"])
    tech_support = st.selectbox("Tech Support", ["No", "Yes", "No internet service"])

with s2:
    online_backup = st.selectbox("Online Backup", ["No", "Yes", "No internet service"])
    device_protection = st.selectbox("Device Protection", ["No", "Yes", "No internet service"])
    streaming_tv = st.selectbox("Streaming TV", ["No", "Yes", "No internet service"])
    streaming_movies = st.selectbox("Streaming Movies", ["No", "Yes", "No internet service"])

st.divider()
st.subheader("Charges")

b1, b2 = st.columns(2)
with b1:
    monthly_charges = st.number_input("Monthly Charges", min_value=0.0, max_value=200.0, value=75.5)
with b2:
    total_charges = st.number_input("Total Charges", min_value=0.0, max_value=10000.0, value=906.0)

st.markdown("</div>", unsafe_allow_html=True)
st.write("")

# ---------------- Build DataFrame ----------------
new_customer = pd.DataFrame({
    "gender": [gender],
    "SeniorCitizen": [senior],
    "Partner": [partner],
    "Dependents": [dependents],
    "tenure": [tenure],
    "PhoneService": [phone_service],
    "MultipleLines": [multiple_lines],
    "InternetService": [internet_service],
    "OnlineSecurity": [online_security],
    "OnlineBackup": [online_backup],
    "DeviceProtection": [device_protection],
    "TechSupport": [tech_support],
    "StreamingTV": [streaming_tv],
    "StreamingMovies": [streaming_movies],
    "Contract": [contract],
    "PaperlessBilling": [paperless],
    "PaymentMethod": [payment],
    "MonthlyCharges": [monthly_charges],
    "TotalCharges": [total_charges]
})

with st.expander("Preview input row"):
    st.dataframe(new_customer, use_container_width=True)

# ---------------- Center Predict Button ----------------
colL, colM, colR = st.columns([1, 2, 1])
with colM:
    predict_clicked = st.button("🔮 Predict", use_container_width=True)

# ---------------- Prediction Result ----------------
if predict_clicked:
    pred = model.predict(new_customer)[0]
    proba = model.predict_proba(new_customer)[0]
    stay_prob = float(proba[0])
    churn_prob = float(proba[1])

    st.write("")
    st.markdown('<div class="result-card">', unsafe_allow_html=True)
    st.markdown('<h3 class="center">Result</h3>', unsafe_allow_html=True)

    r1, r2, r3 = st.columns([1, 2, 1])
    with r2:
        if pred == 1:
            st.error(f"Customer WILL churn")
        else:
            st.success(f"Customer will NOT churn")

        st.write(f"Churn probability: {churn_prob:.2f}")
        st.progress(min(max(churn_prob, 0.0), 1.0))
        st.caption(f"Staying: {stay_prob:.2f}  |  Churn: {churn_prob:.2f}")

    st.markdown("</div>", unsafe_allow_html=True)

Overwriting app.py
